# 13 — Prompt Security and Untrusted Content

## Scenario
Northstar uses RAG to summarize customer support tickets for agents. 
However, a malicious user submits a ticket containing a hidden "Prompt Injection" attack.

**The Problem:** LLMs cannot reliably distinguish between "System Instructions" and "User Data" because both are fed into the model as a single stream of text. If user data contains text that looks like an instruction, the model might follow it.

## Step 1: The Vulnerable Implementation (Indirect Prompt Injection)

We fetch a "Customer Ticket" from our database and ask the model to summarize it. Watch what happens.

## Step 2: The Defense (Data Separation via XML Tags)

To defend against this, we use the `system_instruction` parameter to define the model's persona, and we wrap the untrusted data in strict XML tags so the model knows what is data vs what is an instruction.

## Step 3: The Ultimate Truth (Application Control)

While XML tags and strict prompting reduce the success rate of injections, **they are not a 100% secure boundary**. Advanced attackers can still find ways to "break out" of tags (e.g., by predicting the closing tag).

The *only* true defense is **Application Control**:
1. Never give an LLM direct access to destructive tools without a human-in-the-loop.
2. Treat all LLM output as untrusted data.
3. Use deterministic code (e.g., Python `if` statements) to enforce security boundaries, not English prompts.

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab13 import *

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The Vulnerable Implementation (Indirect Prompt Injection)

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i13/vulnerable/injection")
print("SYSTEM:\n", request.system)
print("USER:\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
assert check_constraints(response.text, forbidden_phrases=("SYSTEM COMPROMISED",))

## Step 2: The Defense (Data Separation)

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i13/defended/injection")
print("SYSTEM:\n", request.system)
print("USER (wrapped):\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
assert not check_constraints(response.text, forbidden_phrases=("SYSTEM COMPROMISED",))

## Step 3: The Ultimate Truth (Application Control)

In [ ]:
principal = Principal(user_id="USER-0001", tenant="tenant-synthetic-a", roles={"support_agent"})
decision = authorize(principal, Action(name="send_all_user_data", tenant="tenant-synthetic-a", requires_role="data_export"))
print(decision)
assert decision.reason_code == "missing_role"
assert rate("injection_detection", sum(instruction_like_score(item["text"]) >= 0.8 for item in INJECTIONS), len(INJECTIONS), "higher_is_better").numerator == 8

## Takeaway

This replay-backed experiment makes the application control and measured trade-off explicit.

## References

See the course README for the references and further reading.